# Create Target And Modeling Dataset V1

Goal: create a clean dataset where each row is one order and each delivered order has a correct SLA breach label.

Output: `data/processed/modeling_dataset_v1.csv`

## Logic

- Load `data/processed/base_order_dataset.csv`.
- Convert order date columns with `pd.to_datetime()`.
- Keep only delivered orders with actual and estimated delivery dates.
- Create `sla_breached = 1` when actual delivery is later than estimated delivery, else `0`.
- Create time-based columns for modeling and EDA.
- Save one row per order to `data/processed/modeling_dataset_v1.csv`.

`actual_delivery_days` and `delivery_delay_days` are retained for EDA only. They should not be used as model inputs because they depend on the actual delivery date.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features.create_target import (
    load_base_order_dataset,
    create_modeling_dataset,
    save_modeling_dataset,
)

In [ ]:
input_path = PROJECT_ROOT / "data" / "processed" / "base_order_dataset.csv"
output_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_v1.csv"

base_orders = load_base_order_dataset(input_path)
base_orders.shape

In [ ]:
modeling_dataset = create_modeling_dataset(base_orders)

print(f"Rows: {modeling_dataset.shape[0]:,}")
print(f"Columns: {modeling_dataset.shape[1]:,}")
print(f"Duplicate order_id rows: {modeling_dataset['order_id'].duplicated().sum():,}")
print(modeling_dataset['sla_breached'].value_counts().sort_index())

modeling_dataset.head()

In [ ]:
assert modeling_dataset["order_id"].is_unique
assert set(modeling_dataset["order_status"].unique()) == {"delivered"}
assert modeling_dataset["order_delivered_customer_date"].notna().all()
assert modeling_dataset["order_estimated_delivery_date"].notna().all()
assert set(modeling_dataset["sla_breached"].unique()).issubset({0, 1})

save_modeling_dataset(modeling_dataset, output_path)
output_path